# Math Photo Solver — Обучение в Google Colab

1. Подключение Google Drive
2. Клонирование / обновление репозитория
3. Установка зависимостей
4. Проверка GPU
5. Генерация датасета символов 150×150 px (80/20)
6. Обучение ResNet-18
7. Оценка точности
8. Сохранение на Google Drive
9. Скачивание модели
10А. Тест солвера напрямую
10Б. Тест OCR на реальном фото
11. Запуск сайта через ngrok

> **Перед запуском**: `Среда выполнения → Сменить тип → GPU (T4) → Сохранить`

## Шаг 1. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/math_solver_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Google Drive подключён:', DRIVE_SAVE_DIR)

## Шаг 2. Клонирование / обновление репозитория
Если репозиторий уже скачан — просто подтягивает последние изменения.

In [ ]:
import os
REPO_URL    = 'https://github.com/sergey2321/sergey2321.git'
BRANCH      = 'claude/ale-yP87K'
REPO_DIR    = '/content/math-solver'

if os.path.exists(REPO_DIR):
    print('Репозиторий уже есть — подтягиваем изменения...')
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git checkout {BRANCH}

# Показать текущий коммит
!git log --oneline -3
print('\nРепозиторий актуален.')

## Шаг 3. Установка зависимостей

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok
print('Зависимости установлены.')

## Шаг 4. Проверка GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ВНИМАНИЕ: GPU не найден. Обучение будет медленным.')

## Шаг 5. Генерация датасета символов (80/20)
Текущие параметры: **150×150 px**, шрифт **40 pt**, 10 000 изображений.

In [ ]:
import os

DATASET_COUNT = 10000   # увеличь до 20000+ для лучшей точности
DATASET_DIR   = 'dataset/symbols'

# Удалить старый датасет если он был другого размера
if os.path.exists(DATASET_DIR):
    import shutil
    shutil.rmtree(DATASET_DIR)
    print('Старый датасет удалён.')

!python -m dataset.generator.generate_handwritten \
    --count {DATASET_COUNT} --out {DATASET_DIR} --seed 42

train_n = len(os.listdir(f'{DATASET_DIR}/train'))
val_n   = len(os.listdir(f'{DATASET_DIR}/val'))
print(f'Train: {train_n}  |  Val: {val_n}  |  Всего: {train_n + val_n}')

### Проверка параметров датасета

In [ ]:
import json, random
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

with open(f'{DATASET_DIR}/metadata.json') as f:
    meta = json.load(f)

train, val = meta['train'], meta['val']
total = len(train) + len(val)

print(f'Всего изображений : {total}')
print(f'Train (80%)       : {len(train)}')
print(f'Val   (20%)       : {len(val)}')
print(f'Классов           : {len(meta["symbols"])}')
print(f'Символы           : {", ".join(meta["symbols"])}')

# Проверяем размер изображений
sample_img = Image.open(f"{DATASET_DIR}/{train[0]['file']}")
print(f'Размер изображения: {sample_img.size[0]}×{sample_img.size[1]} px')

# Показываем по 3 примера каждого символа
by_class = {s: [] for s in meta['symbols']}
for r in train:
    by_class[r['symbol']].append(r)

cols = len(meta['symbols'])
fig, axes = plt.subplots(3, cols, figsize=(cols * 1.5, 5))
fig.patch.set_facecolor('#0b0d14')
for col, sym in enumerate(meta['symbols']):
    samples = random.sample(by_class[sym], 3)
    for row, rec in enumerate(samples):
        ax = axes[row][col]
        ax.set_facecolor('#1a1d27')
        img = Image.open(f"{DATASET_DIR}/{rec['file']}")
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if row == 0:
            ax.set_title(sym, fontsize=8, color='#8b96ff', pad=2)
plt.suptitle('3 примера каждого символа (150×150 px, шрифт 40 pt)', color='white', fontsize=11)
plt.tight_layout()
plt.show()

## Шаг 6. Обучение модели

In [ ]:
EPOCHS     = 25
BATCH_SIZE = 64    # уменьшен с 128 т.к. изображения стали больше (150×150)
LR         = 1e-3
MODEL_OUT  = 'backend/models/symbol_clf.pth'

!python -m training.train_ocr \
    --data {DATASET_DIR} --epochs {EPOCHS} \
    --batch {BATCH_SIZE} --lr {LR} --out {MODEL_OUT}

## Шаг 7. Оценка точности модели

In [ ]:
!python -m training.evaluate --data {DATASET_DIR} --model {MODEL_OUT}

## Шаг 8. Сохранение модели на Google Drive

In [ ]:
import shutil
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
shutil.copy(MODEL_OUT, f'{DRIVE_SAVE_DIR}/symbol_clf_{ts}.pth')
shutil.copy(MODEL_OUT, f'{DRIVE_SAVE_DIR}/symbol_clf_latest.pth')
print(f'Сохранено: {DRIVE_SAVE_DIR}/symbol_clf_latest.pth')

## Шаг 9. Скачивание модели
**Способ 1** — раскомментируй ячейку.  
**Способ 2** — Google Drive: `math_solver_models/symbol_clf_latest.pth`

In [ ]:
# from google.colab import files
# files.download('backend/models/symbol_clf.pth')

---
## Шаг 10А. Тест солвера НАПРЯМУЮ (без OCR)
Выражение уже известно — OCR не нужен. Тестируем математический движок.

In [ ]:
import sys, io
sys.path.insert(0, '/content/math-solver')

import matplotlib.pyplot as plt
from PIL import Image as PILImage
from backend.solver.solver import solve
from backend.utils.image_generator import render_expression_image

TEST_EXPRESSIONS = [
    '2 + 2',
    '15 * 3 - 7',
    '2*x + 3 = 7',
    'x**2 - 5*x + 6 = 0',
    'integrate(x**2, x)',
    'diff(x**3 + 2*x, x)',
    '(x + 1)**2',
    '2*x**2 + 3*x - 2 = 0',
]

# Показываем сгенерированные картинки
fig, axes = plt.subplots(len(TEST_EXPRESSIONS), 1, figsize=(10, len(TEST_EXPRESSIONS) * 1.3))
fig.patch.set_facecolor('#0b0d14')
for i, expr in enumerate(TEST_EXPRESSIONS):
    png = render_expression_image(expr)
    img = PILImage.open(io.BytesIO(png))
    axes[i].imshow(img)
    axes[i].axis('off')
plt.suptitle('Тестовые задачи', color='white', fontsize=13)
plt.tight_layout()
plt.show()

# Решаем напрямую
print('=' * 65)
print('ТЕСТ СОЛВЕРА (прямое решение, без OCR)')
print('=' * 65)
ok_count = 0
for i, expr in enumerate(TEST_EXPRESSIONS):
    r = solve(expr)
    mark = '✓' if not r['error'] else '✗'
    if not r['error']:
        ok_count += 1
    print(f'  {mark} [{i+1}] {expr}')
    print(f'       Ответ: {r["answer"]}')
    if r['error']:
        print(f'       Ошибка: {r["error"]}')
print(f'\nРезультат: {ok_count}/{len(TEST_EXPRESSIONS)}')

---
## Шаг 10Б. Тест OCR на реальном фото
Загрузи фото с задачей — система его распознает и решит.

In [ ]:
import sys, io
sys.path.insert(0, '/content/math-solver')

from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
from backend.preprocessing.image_prep import preprocess_image
from backend.ocr.printed import extract_expression
from backend.solver.solver import solve

print('Загрузи фото с математической задачей:')
uploaded = files.upload()

for filename, img_bytes in uploaded.items():
    print(f'\nФайл: {filename}')
    img = Image.open(io.BytesIO(img_bytes))
    plt.figure(figsize=(8, 3))
    plt.imshow(img); plt.axis('off'); plt.title('Загруженное фото')
    plt.show()

    preprocessed = preprocess_image(img_bytes)
    recognized   = extract_expression(preprocessed)
    print(f'Распознано : "{recognized}"')

    if not recognized.strip():
        print('OCR ничего не распознал. Попробуй более чёткое фото.')
        continue

    result = solve(recognized)
    print(f'Ответ    : {result["answer"]}')
    print(f'Проверка : {"✓ пройдена" if result["verified"] else "✗ не пройдена"}')
    if result['error']:
        print(f'Ошибка   : {result["error"]}')
    print('Шаги:')
    for step in result['steps']:
        print(f'  {step}')

---
## Шаг 11. Запуск сайта через ngrok
> Токен: [ngrok.com](https://ngrok.com) → `Your Authtoken`

In [ ]:
NGROK_TOKEN = 'ВСТАВЬ_ТОКЕН_СЮДА'

from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = NGROK_TOKEN
server = subprocess.Popen(
    ['uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/math-solver',
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)
public_url = ngrok.connect(8000)
print('=' * 50)
print(f'  Сайт: {public_url}')
print('=' * 50)

In [ ]:
ngrok.disconnect(public_url)
server.terminate()
print('Сервер остановлен.')

---
## Бонус: Визуализация датасета

In [ ]:
import json, random
import matplotlib.pyplot as plt
from PIL import Image

with open(f'{DATASET_DIR}/metadata.json') as f:
    meta = json.load(f)

samples = random.sample(meta['train'], min(20, len(meta['train'])))
fig, axes = plt.subplots(2, 10, figsize=(20, 5))
fig.patch.set_facecolor('#0b0d14')
for ax, rec in zip(axes.flat, samples):
    img = Image.open(f"{DATASET_DIR}/{rec['file']}")
    ax.imshow(img, cmap='gray')
    ax.set_title(rec['symbol'], fontsize=10, color='#8b96ff')
    ax.axis('off')
    ax.set_facecolor('#1a1d27')
plt.suptitle('Примеры символов из датасета', color='white', fontsize=14)
plt.tight_layout()
plt.show()